## Parametrization and Channel Generation

In [ ]:
import sys; sys.path.append('../../../'); sys.path.append('../../../periodic_patches/'); sys.path.append('../../experiments/'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
from py_newton_optimizer import NewtonOptimizerOptions

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(32)
parallelism.set_gradient_assembly_num_threads(32)
parallelism.set_hessian_assembly_num_threads(32)

In [ ]:
import utils, mesh_utilities
importlib.reload(utils)

In [ ]:
target_surf = mesh.Mesh("../../../../examples/igloo.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
# target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
lines = np.array([[-1.        , -1.        ,  2.415     ],
       [-2.30769231, -1.        ,  3.81923077],
       [ 0.77304965,  1.        , -2.27304965],
       [ 1.29357798,  1.        , -2.94036697],
       [-0.43333333, -1.        ,  1.655     ]])

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

for i in range(1000): lg.runIteration()

# lg.alphaMin = 1.354641650129586
# lg.alphaMax = 1.3936682623611498

# lg.betaMin = 1.1282764273066557
# lg.betaMax = 1.141976581731514

# lg.alphaMin = 1.2349791815955107 #1.354641650129586
# lg.alphaMax = 1.2827966225656238 #1.3936682623611498

# lg.betaMin = 1.2349791815955107 #1.1282764273066557
# lg.betaMax = 1.2827966225656238 #1.141976581731514


lg.setLines(lines)
lg.alphaMin = 1.0
lg.alphaMax = 1.5

lg.betaMin = 1.0
lg.betaMax = 1.5


(1.3936682623611498, 1.354641650129586, 1.141976581731514, 1.1282764273066557)

print(lg.energy())
lg.runIteration()
print(lg.energy())

for i in range(5000): lg.runIteration()
print(lg.energy())

In [ ]:
np.set_printoptions(suppress=True)

In [ ]:
lg.leftStretchAngles()

In [ ]:
visualization.visualize_both(lg)

In [ ]:
lg.energy()

### Get Splines

In [ ]:
grid_data = np.load("../../Visualization/grid_data.npy")[:, :7, :7]
grid_pattern_1 = np.load("../../Visualization/grid_pattern_1.npy")[:7]
grid_pattern_2 = np.load("../../Visualization/grid_pattern_2.npy")[:7]

In [ ]:
import parametrization_helper

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:
grid_data.shape

In [ ]:
grid_pattern_1.shape

In [ ]:
splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(grid_data, grid_pattern_1, grid_pattern_2)

In [ ]:
splines[0]([0, 1])

In [ ]:
default_pattern_params = [0.5 * (np.max(grid_pattern_1) + np.min(grid_pattern_1))] * len(lg.getAlphas()) + [0.5 * (np.max(grid_pattern_2) + np.min(grid_pattern_2))] * len(lg.getAlphas())

In [ ]:
len(grid_data.shape) - 1

In [ ]:
len(default_pattern_params)

In [ ]:
mat_info = np.array(default_pattern_params).reshape((2, len(lg.getAlphas())))

In [ ]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params, len(grid_data.shape) - 1)
rparam.patternParamBounds = np.array([[0.5, 2.4], [0, 90]])

In [ ]:
visualization.visualize_both(rparam, height = 4, showBarriers=True)

In [ ]:
rparam.

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType

In [ ]:
rparam.energy(PET.RGP)

In [ ]:
rparam.energy(PET.Bending)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.RGP, PET.Bending]))

In [ ]:
def optimize_rparam(param, patternRegW, phiRegW, bendRegW = 0.0, update_uv = True, niter = 100):
    param.patternRegW = patternRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    param.diffRegW = 0.0
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = niter
    opts.gradTol = 1e-9
    opts.factorizer = opts.factorizer.CatamariNesdis
    benchmark.reset()
    
    if update_uv:
        fixedvars = [param.uOffset(), param.vOffset(), param.phiOffset()]
    else:
        fixedvars = range(param.stretchOffset())

    cr = parametrization.pattern_parametrization_knitro(param, opts.niter, fixedvars)
    benchmark.report()
    return cr

In [ ]:
rparam.bendRegW = 1e-4

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, 0, 0, bendRegW = 0, update_uv = False, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
importlib.reload(visualization)
visualization.visualize_pattern(rparam, height = 4, showBarriers=False)

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 1e-3, bendRegW = 1e-1, update_uv = True, niter = 500)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
importlib.reload(visualization)
visualization.visualize_pattern(rparam, height = 4, showBarriers=False)

In [ ]:
parametrization_helper.visualize_scale_factors(lines, rparam.getAlphas(), rparam.getBetas())

In [ ]:
importlib.reload(visualization)

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False)

In [ ]:
# visualization.singularValueHistogramBoth(rparam)

## Upsampling and channel generation

In [ ]:
nsubdiv=11
upsampledMesh, upsampledAngles, upsampledPatternParams = rparam.upsampledVertexLeftStretchAnglesAndPatternParameters(nsubdiv)


In [ ]:
radius_data =  upsampledPatternParams[0]
radius_data = radius_data / 2.5 * (np.pi / 2)

angles_data =  upsampledPatternParams[1]
angles_data = angles_data / 180 * np.pi

In [ ]:
(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_cross_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles, radius_data, angles_data, frequency=0.2, margin = 0.07)


# pickle.dump((sdfVertices, sdfTris, sdf), open('stripe_sdf_ns4_f100.pkl', 'wb'))

# import pickle, mesh, wall_generation, visualization, numpy as np
# (sdfVertices, sdfTris, sdf) = pickle.load(open('stripe_sdf_ns4_f100.pkl', 'rb'))

importlib.reload(visualization)

import matplotlib as mpl

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 10, height=10)

In [ ]:
importlib.reload(visualization)
visualization.scalarFieldPlotZeroContourFast(sdfVertices, sdfTris, sdf, width = 15, height=10, cmap = mpl.colormaps["PiYG"])

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=1,
                                              minContourLen=1)

visualization.plot_line_segments(pts, edges, width=15, height=15)

## Meshing and inflation simulation

In [ ]:
import sheet_meshing, inflation


In [ ]:
m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, pts, edges, triArea=1e0)


In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(iwv) == 1)[0], width=10, height=10)


In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, np.array(iwv) != 0)

In [ ]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [ ]:
isheet.pressure = 1e-5

In [ ]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [ ]:
isheet.pressure = 1e-2

In [ ]:
opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

In [ ]:
# name = "igloo_with_bending"
# time_stamp = time.strftime("%Y_%m_%d_%H_%M")
# result_folder = 'output/{}/{}'.format(name, time_stamp)
# if not os.path.exists(result_folder):
#     os.makedirs(result_folder)  

# variable = "frequency=0.2_margin=0.07_targetEdgeSpacing=1_minContourLen=1_triArea=1e0_isheet.pressure=1e-1"

# m.save("{}/mesh_{}_{}.obj".format(result_folder, name, variable))


# np.save("{}/fusedVtx_{}_{}.npy".format(result_folder, name, variable), iwv)

# np.save("{}/liftedSheetPositions_{}_{}.npy".format(result_folder, name, variable), liftedSheetPositions)

# np.save("{}/dofs{}_{}.npy".format(result_folder, name, variable), isheet.getVars())

## Shape Optimization

In [ ]:
# Reset the inflation and set up target-attraction forces
# isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, target_surf)
targetAttractedSheet.energy(targetAttractedSheet.EnergyType.Fitting)

In [ ]:
targetAttractedSheet.targetSurfaceFitter().holdClosestPointsFixed = True
targetAttractedSheet.fittingWeight = 1e-5

In [ ]:
importlib.reload(visualization)

In [ ]:
target_view = visualization.ViewerWithSurface(targetAttractedSheet, target_surf, width=768, height=640)

In [ ]:
target_view.showWireframe(True)
visualization.set_surface_view_options(target_view, sheet_transparent=False, surface_transparent=True,  surface_color = 'lightgray', color = 'green')

target_view.show()

In [ ]:
targetAttractedSheet.sheet().pressure = 1e-1

In [ ]:
targetAttractedSheet.sheet().setVars(isheet.getVars())

In [ ]:
framerate = 5
def cb(it):
    if it % framerate == 0:
        # target_view.update(scalarField=utils.getStrains(targetAttractedSheet.sheet())[:, 0])    
        target_view.update()

opts.niter = 100

import time
benchmark.reset()
cr = inflation.inflation_newton(targetAttractedSheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

In [ ]:
# Set up the sheet optimizer
import sheet_optimizer, opt_config
origDesignMesh = isheet.mesh().copy()

sheet_opt = sheet_optimizer.PySheetOptimizer(targetAttractedSheet, fixedVars, renderMode=sheet_optimizer.RenderMode.PYTHREEJS,
                                             detActivationThreshold=0.9, detActivationThresholdTubeTri=0.5,
                                             originalDesignMesh=origDesignMesh, fusingCurveSmoothnessConfig=opt_config.FusingCurveSmoothnessParams(0.0, 0.0, 1.0, 1.0))

In [ ]:
# Configure some more weights
sheet_opt.rso.compressionPenaltyWeight = 1e-6
fcs = sheet_opt.rso.fusingCurveSmoothness()
fcs.interiorWeight = 0.05

In [ ]:
sheet_opt.flat_viewer.showWireframe()
sheet_opt.viewer()

In [ ]:
# Run the optimization
sheet_opt.setSolver(sheet_optimizer.Solver.SCIPY)
sheet_opt.optimize()

In [ ]:
# Lower the interior weight
fcs = sheet_opt.rso.fusingCurveSmoothness()
fcs.interiorWeight = 0.05

In [ ]:
# Continue the optimization
sheet_opt.optimize()

In [ ]:
utils.allGradientNorms(sheet_opt.rso)

In [ ]:
utils.allEnergies(sheet_opt.rso)

In [ ]:
# Remove the target-attraction force and recompute the equilibrium
targetAttractedSheet.fittingWeight = 1e-8
inflation.inflation_newton(targetAttractedSheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
viewer.update()

In [ ]:
# Save the full state for later reloading with `sheet_optimizer.load()`
# sheet_opt.save('igloo_with_bending_sheet_opt.pkl.gz')

### Generate Fabrication Files

In [ ]:
import sheet_optimizer, opt_config

In [ ]:
sheet_opt = sheet_optimizer.load('igloo_with_bending_sheet_opt.pkl.gz')

In [ ]:
scaleFactor = 1.15 # Factor for fine-tuning size to fit the machine's build area
channelMargin = 8 / scaleFactor # 8mm channel margin
tabMargin = 2 / scaleFactor # 2mm tab margin

In [ ]:
isheet = sheet_opt.rso.sheet()
optMesh = sheet_opt.rso.mesh().copy()
origMesh = sheet_opt.rso.originalMesh().copy()
import inflation
tas = sheet_opt.rso.targetAttractedInflation()
tsf = tas.targetSurfaceFitter()
targetSurf = mesh.Mesh(tsf.targetSurfaceV, tsf.targetSurfaceF)
iwv = [isheet.isWallVtx(i) for i in range(isheet.mesh().numVertices())]

In [ ]:
uv = rparam.uv()

In [ ]:
!pip install shapely==1.7.0

In [ ]:
importlib.reload(fabrication)
importlib.reload(shapely)
import shapely.geometry as shp


In [ ]:
shapely.__version__

In [ ]:
import fabrication
fabrication.writeFabricationData('fabrication_data/igloo/with_bending', origMesh, optMesh, iwv, targetSurf, uv,
                                 scale=scaleFactor, numTabs=80, inletOffset=0.742, tabOffset=0.60 / 80,
                                 channelMargin=channelMargin, tabMargin=tabMargin, tabWidth=5, tabHeight=8, fuseSeamWidth=1.0, inletScale=12 / channelMargin / scaleFactor,
                                 overlap=0.0, smartOuterChannel=True)